In [ ]:
#STEP 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#STEP 2 — Import Required Libraries
import pandas as pd

In [ ]:
#STEP 3 — Define File Paths
rf_path = "/content/drive/MyDrive/FINAL_RF_variant_predictions_All_20_Cases.csv"

svm_path = "/content/drive/MyDrive/SVM_final_variant_results_20_cases.csv"

xgb_path = "/content/drive/MyDrive/XGBoost_final_variant_predictions.csv"

In [ ]:
#STEP 4 — Load the Files
rf = pd.read_csv(rf_path)
svm = pd.read_csv(svm_path)
xgb = pd.read_csv(xgb_path)

print("RF shape:", rf.shape)
print("SVM shape:", svm.shape)
print("XGB shape:", xgb.shape)

RF shape: (625, 57)
SVM shape: (12344, 55)
XGB shape: (1093, 55)


In [ ]:
#STEP 5 — Create Variant ID
rf_unique = rf.drop_duplicates(subset="variant_id")
svm_unique = svm.drop_duplicates(subset="variant_id")
xgb_unique = xgb.drop_duplicates(subset="variant_id")

In [ ]:
#STEP 6 — Filter Variants Predicted Pathogenic
rf_pathogenic = rf[rf["RF_prediction"] == 1]

svm_pathogenic = svm[svm["SVM_prediction"] == 1]

xgb_pathogenic = xgb[xgb["XGB_prediction"] == 1]

print("RF pathogenic:", rf_pathogenic.shape)
print("SVM pathogenic:", svm_pathogenic.shape)
print("XGB pathogenic:", xgb_pathogenic.shape)

RF pathogenic: (142, 58)
SVM pathogenic: (6963, 56)
XGB pathogenic: (181, 56)


In [ ]:
#STEP 7 — Find Variants Shared by All 3 Models
common_variants = set(rf_pathogenic["variant_id"]) & \
                  set(svm_pathogenic["variant_id"]) & \
                  set(xgb_pathogenic["variant_id"])

print("Variants predicted by all three models:", len(common_variants))

Variants predicted by all three models: 2


In [ ]:
#STEP 8 — Extract Consensus Biomarkers
# Remove duplicate variants
rf_unique = rf.drop_duplicates(subset="variant_id")
svm_unique = svm.drop_duplicates(subset="variant_id")
xgb_unique = xgb.drop_duplicates(subset="variant_id")

# Keep only prediction columns
svm_small = svm_unique[["variant_id","SVM_prediction","SVM_probability"]]
xgb_small = xgb_unique[["variant_id","XGB_prediction","XGB_probability"]]

# Merge models
merged = rf_unique.merge(svm_small,on="variant_id",how="inner")
merged = merged.merge(xgb_small,on="variant_id",how="inner")

# Extract consensus biomarkers
consensus_variants = merged[
    (merged["RF_prediction"] == 1) &
    (merged["SVM_prediction"] == 1) &
    (merged["XGB_prediction"] == 1)
]

print("Consensus biomarker variants:", consensus_variants.shape)

Consensus biomarker variants: (2, 62)


In [ ]:
#STEP 9 — Save the Final Biomarker File
consensus_variants.to_csv("Final_High_Confidence_Potential_Biomarkers.csv", index=False)

print("File saved successfully")

File saved successfully


In [ ]:
#Download the File

from google.colab import files

files.download("Final_High_Confidence_Potential_Biomarkers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>